# 12 — Direction 1: the replica against the original (step C; now the regression check)

`hvac_controller/d1/env_check.py`, one process per scenario. For each scenario it builds the ROM (step B) and the
exogenous series (weather, occupancy, schedule, tariffs, time features) from the organisers' own objects, then for each
policy traces the real environment and compares the replica (`torch_env.BuildingEnv` = ROM + `plant.py`) tier by tier:

| tier | what | target |
|---|---|---|
| T0 | exogenous series vs the trace | exact |
| T1 | plant equations fed the traced state (teacher-forced): q, g, Ts, flows, hot water, tank, billed energy, reward terms, observation | rounding (≤ 1e-9 rel; float32 fields to 1 ulp) |
| T3 | replica alone on the traced actions (open loop) | zone max < 0.05 K, reward sum < 0.5 % |
| T4 | the same policy on the replica's own observations (closed loop) | score and terms; chaotic in hot/humid climates (D1 doc §2.7), judged by noise ensembles (`d1/evaluate.py`) |
| T5 | torch float64/float32 vs NumPy, gradcheck (relaxed window), 288-step backward | torch64 = NumPy; gradients finite |

Step C passed on all 30 scenarios (2026-10-18). Rerun `debug` after any change to `plant.py`, `rom.py`,
`torch_env.py` or `spec.py`: T1 must stay 0 and T3 at its old values (zone ≤ 0.0014 K).

**MODE = "debug"**: 3 scenarios × 2 days × (pid_water, bang_bang, excite), ≈ 20–30 min.
**MODE = "gate"**: all 30 scenarios, pid_water for the full 30 days plus excite for 2 days; run `N_SHARDS` sessions in
parallel with `RUN_ID = 0 … N_SHARDS-1` (each takes every N_SHARDS-th scenario).

**Setup:** Internet on (clones the repo); attach the competition data. CPU. Output: `/kaggle/working/d1_env_<mode>_<run>.zip`.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
MODE = "debug"   # "debug" or "gate"
RUN_ID, N_SHARDS = 0, 3   # gate
N_JOBS = 3       # scenarios in parallel (4 CPUs)
DEBUG_SCENARIOS = ["building_train_1/atlanta_2", "building_train_2/phoenix_6", "building_train_8/fairbanks_8"]

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

REPO = Path("/tmp/hvac-controller")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

SHIPPED = next(p for p in sorted(Path("/kaggle/input").rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
shutil.copytree(SHIPPED, REPO / "smart_buildings" / "training_scenarios", dirs_exist_ok=True)
ALL = sorted(f"{p.parent.parent.name}/{p.parent.name}"
             for p in (REPO / "smart_buildings" / "training_scenarios").glob("*/*/config.gin"))
if MODE == "debug":
    SCENARIOS, POLICIES, STEPS = DEBUG_SCENARIOS, "pid_water,bang_bang,excite", "576,576,576"
elif MODE == "gate":
    SCENARIOS, POLICIES, STEPS = ALL[RUN_ID::N_SHARDS], "pid_water,excite", "8640,576"
else:
    raise ValueError(MODE)
assert set(SCENARIOS) <= set(ALL), sorted(set(SCENARIOS) - set(ALL))
print(len(ALL), "scenarios;", len(SCENARIOS), "here:", SCENARIOS)

In [ ]:
OUT = Path(f"/kaggle/working/d1_env_{MODE}_{RUN_ID}")
OUT.mkdir(parents=True, exist_ok=True)
threads = str(max(1, (os.cpu_count() or 4) // N_JOBS))
env = dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS=threads, OPENBLAS_NUM_THREADS=threads,
           MKL_NUM_THREADS=threads)
tags = [s.replace("/", "__") for s in SCENARIOS]
t0 = time.time()
pending, running = list(zip(SCENARIOS, tags)), []
while pending or running:
    while pending and len(running) < N_JOBS:
        scenario, tag = pending.pop(0)
        (OUT / tag).mkdir(exist_ok=True)
        log = open(OUT / tag / "log.txt", "w")
        cfg = REPO / "smart_buildings" / "training_scenarios" / scenario / "config.gin"
        cmd = [sys.executable, "-m", "hvac_controller.d1.env_check", str(OUT / tag), "--config", str(cfg),
               "--policies", POLICIES, "--steps", STEPS, "--torch-steps", "576"]
        running.append((tag, subprocess.Popen(cmd, cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT), log))
        print(f"[{time.time() - t0:5.0f}s] started {tag}", flush=True)
    for item in list(running):
        tag, proc, log = item
        if proc.poll() is not None:
            log.close()
            running.remove(item)
            print(f"[{time.time() - t0:5.0f}s] {tag} exit {proc.returncode}", flush=True)
    time.sleep(5)
for tag in tags:
    text = (OUT / tag / "log.txt").read_text()
    print(f"===== {tag}")
    print(text[-3000:] if "Traceback" not in text else text[-8000:])

In [ ]:
reports = {}
for tag in tags:
    f = OUT / tag / "report.json"
    reports[tag] = json.loads(f.read_text()) if f.exists() else {"error": "no report (see log above)"}
(OUT / "reports.json").write_text(json.dumps({"commit": COMMIT, "mode": MODE, "run_id": RUN_ID, "reports": reports},
                                             indent=1))
with zipfile.ZipFile(f"/kaggle/working/d1_env_{MODE}_{RUN_ID}.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(OUT.parent))
print(f"wrote /kaggle/working/d1_env_{MODE}_{RUN_ID}.zip")


def worst(d):
    """Largest max_abs among the error entries of a tier."""
    vals = [(k, v["max_abs"]) for k, v in d.items() if isinstance(v, dict) and "max_abs" in v]
    return max(vals, key=lambda kv: kv[1]) if vals else ("-", 0.0)


gate = []
for tag, r in reports.items():
    if "policies" not in r:
        print(tag, "ERROR", r.get("error"))
        continue
    print(f"\n{tag}: Z={r.get('n_zones')} V={r.get('n_vavs')} A={r.get('n_ahus')} obs={r.get('n_obs')} "
          f"r={r.get('rom_order')} h={r.get('h')} (ROM {r.get('rom_build_s', 0):.0f}s)")
    for name, p in r["policies"].items():
        t0_, t1, t3, t4, t5 = p["T0"], p["T1"], p["T3"], p["T4"], p["T5"]
        k0, e0 = worst(t0_)
        k1, e1 = worst({k: v for k, v in t1.items() if k != "decode"})
        kd, ed = worst(t1["decode"])
        print(f"  {name:>10} {p['steps']:5d} steps | T0 worst {k0} {e0:.1e} | T1 worst {k1} {e1:.1e} "
              f"(obs worst field {t1['obs_worst_field']}), decode worst {kd} {ed:.1e}")
        print(f"  {'':>10}             | T1 q {t1['q_zone']['rel']:.1e} g {t1['g']['rel']:.1e} ts {t1['ts']['max_abs']:.1e} "
              f"tank {t1['tank']['max_abs']:.1e} ret {t1['ret']['max_abs']:.1e} blower {t1['blower']['rel']:.1e} "
              f"cool {t1['cool']['rel']:.1e} gas {t1['gas']['rel']:.1e} pump {t1['pump']['rel']:.1e} "
              f"reward {t1['reward']['max_abs']:.1e} obs {t1['obs']['max_abs']:.1e} obs0 {t1['obs0']['max_abs']:.1e}")
        print(f"  {'':>10}             | T3 zone max {t3['zone_max_k']:.4f} K mae {t3['zone_mae_k']:.5f} | reward sum "
              f"{t3['reward_sum_rel']:.2e} elec {t3['elec_sum_rel']:.2e} gas {t3['gas_sum_rel']:.2e} regret "
              f"{t3['regret_sum_rel']:.2e} | obs {t3['obs']['max_abs']:.1e}")
        print(f"  {'':>10}             | T4 score true {t4['score_true']:.5f} replica {t4['score_replica']:.5f} "
              f"rel {t4['score_rel']:.2%} (zone max vs true {t4['zone_max_k_vs_true']:.3f} K)")
        pt, pr = t4["parts_true"], t4["parts_replica"]
        print(f"  {'':>10}             | T4 parts regret {pt['regret']:.5f}/{pr['regret']:.5f} cost {pt['cost']:.5f}/"
              f"{pr['cost']:.5f} carbon {pt['carbon']:.5f}/{pr['carbon']:.5f} (true/replica)")
        if "obs_src_mae" in t3:
            print(f"  {'':>10}             | T3 replica obs error on state fields: mae {t3['obs_src_mae']:.1e} p99 "
                  f"{t3['obs_src_p99']:.1e} max {t3['obs']['max_abs']:.1e}")
        if "skipped" in t5:
            print(f"  {'':>10}             | T5 {t5['skipped']}")
        else:
            print(f"  {'':>10}             | T5 f64 reward {t5['float64']['reward_vs_numpy']['max_abs']:.1e} "
                  f"f32 reward {t5['float32']['reward_vs_numpy']['max_abs']:.1e} zone {t5['float32']['zone_vs_numpy_k']:.1e} K | "
                  f"gradcheck {t5['gradcheck']} | backward {t5['long_backward']['steps']} steps finite "
                  f"{t5['long_backward']['grad_finite']} {t5['long_backward']['seconds']:.1f}s")
        if name == "pid_water":
            gate.append((tag, t4["score_rel"]))
if gate:
    bad = [g for g in gate if g[1] >= 0.02]
    print(f"\nT4 pid_water closed-loop score within 2 % (informative: chaotic climates move by up to 6.6 %): {len(gate) - len(bad)}/{len(gate)};"
          f" worst {max(g[1] for g in gate):.2%}" + (f"; failing {bad}" if bad else ""))